# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. The dataset includes clinicopathological and molecular data for cancer survivors with second primary colorectal cancer (CRC), annotated using a Croissant schema.

### Dataset Source
The dataset Croissant schema is available at:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (remove the exclamation mark if inside a shell/JupyterLab with internet access)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. We'll point to the Croissant schema URL to access both metadata and data files.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL (Croissant JSON-LD)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's examine the available record sets, their fields, and their corresponding `@id` identifiers provided in the Croissant schema.

- **Record Sets**: Logical collections of records; analogous to tables in tabular datasets.
- **Fields**: Data elements (columns) defined by their `@id` in Croissant.
    
Below, we list all available record sets defined in this dataset and review their fields by their `@id`.

In [ ]:
# List all record sets available in the Croissant schema
from typing import List, Dict

record_sets = list(dataset.record_sets.keys())
print('Available record sets (by @id):')
for rs_id in record_sets:
    print(f"  - {rs_id}")
    rs = dataset.record_sets[rs_id]
    # List fields by @id
    print("    Fields:")
    for field in rs.fields.values():
        print(f"      - {field.id} (name: {field.name})")
    print()

## 3. Data Extraction
We will load records from each record set into pandas DataFrames for downstream analysis, referencing record set and field `@id`s.

You can select the record set and field you are interested in by their `@id` as listed above.

In [ ]:
# Extract data from all record sets into DataFrames
dfs = dict()
for rs_id in record_sets:
    print(f"Loading record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(records)
    print(f"  Columns: {list(dfs[rs_id].columns)}\n  Rows: {len(dfs[rs_id])}")
    if len(dfs[rs_id]) > 0:
        display(dfs[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform basic data analysis and processing steps on the main tabular record set.

- **Selecting a numeric field**: Typically, demographic or biomarker data (e.g., age).
- **Filtering records**: Filter rows on numeric criteria.
- **Normalizing numeric fields**: Standardize numerical data for further analysis.
- **Grouping and summarizing**: Summarize data by categorical variables, e.g., sex or MSI status.

All field and record set references use their Croissant `@id`.

In [ ]:
# Select the primary tabular record set (replace with your dataset's main @id, taken from section 2)
# For this dataset, likely choices are like 'https://api.app.sen.science/frontiers/7862866/629c16ec-37ee-4556-a351-d5164116c2dd/recordSet/clinical_data' or similar.
# Let's use the first record_set for demonstration, update the @id if your dataset differs!
record_set_id = record_sets[0]
df = dfs[record_set_id]

print(f"Using record set: {record_set_id}")
print("Columns:", df.columns.tolist())

# Heuristically select a numeric field (commonly 'age', 'interval', or similar)
import re
numeric_field_candidates = [col for col in df.columns if re.search(r'age|interval|duration|count|years|score|number|size', col, re.IGNORECASE)]
numeric_field_id = numeric_field_candidates[0] if len(numeric_field_candidates) else df.select_dtypes(include='number').columns[0]
print(f"Chosen numeric field for EDA: {numeric_field_id}")

# Filter: show records with the field above its median value
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (z-score):")
    print(filtered_df[[numeric_field_id, field_norm]].head())

    # Try to group by a likely categorical field
    # Look for reasonable choices such as 'sex', 'msi', 'status', or 'category'
    cat_field_candidates = [col for col in df.columns if re.search(r'sex|gender|msi|status|type|category|group', col, re.IGNORECASE)]
    if cat_field_candidates:
        group_field = cat_field_candidates[0]
        if group_field in filtered_df.columns:
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nMean {numeric_field_id} by {group_field}:")
            print(grouped)
else:
    print(f"The chosen field '{numeric_field_id}' is not numeric.")

## 5. Visualization
Here we visualize the distribution of our chosen numeric field and, if available, its distribution by a categorical field (e.g., sex, MSI status, anatomical type).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(6,3))
sns.histplot(df[numeric_field_id], kde=True, bins=15, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.tight_layout()
plt.show()

# If categorical field is available (from EDA above), plot boxplot
if 'group_field' in locals() and group_field in df.columns:
    plt.figure(figsize=(7,3))
    sns.boxplot(data=df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and inspect a dataset defined with the [Croissant schema](https://mlcommons.org/croissant/), using the `mlcroissant` Python library
- Reference all data entities strictly by their record set and field `@id` values for reproducibility and clarity
- Extract and analyze tabular data: filter, normalize, group by field, and visualize distributions

This methodology ensures transparent, schema-agnostic data science workflows for FAIR datasets as required by the MLCommons Croissant ecosystem.